# Molecular Dynamics Analysis

Parameters (papermill-injected)

In [ ]:
tpr_file = "step5_1.tpr"
xtc_file = "step5_1.xtc"
ndx_input = "index_new.ndx"
ndx_updated = "index_updated.ndx"
gro_file = "step5_1.gro"
protein_resids = [93, 109, 110, 170, 263, 171, 86, 262, 147, 445, 261, 318, 108]
ligand_resnames = ["4GA", "0GA", "ROH", "PHO"]
water_resname = "WAT"
cutoff_occ = 10.0
hbond_groups = "25 36"
pbc_selection = "4 0"
make_ndx_commands = """
1 | 14
r 170
r 263
r 109
r 110
18 | 19
16 | 17
r 14 15 22 86 90 93 97 108 109 110 111 115 118 145 146 147 148 149 168 169 170 171 172 173 174 175 177 178 195 199 200 201 202 204 228 238 242 243 244 245 246 247 248 260 261 262 263 264 265 266 273 276 277 281 283 284 286 287 291 314 315 316 317 318 319 321 357 445
name 36 pocket
r 93
27 | 29
name 38 complex
q
"""

import subprocess

## PBC and Fit Processing

In [ ]:
subprocess.run(
    f'echo "{pbc_selection}" | gmx trjconv -s {tpr_file} -f {xtc_file} -n {ndx_input} -pbc mol -center -ur compact -o traj_pbc.xtc',
    shell=True, check=True
)
subprocess.run(
    f'echo "{pbc_selection}" | gmx trjconv -s {tpr_file} -f traj_pbc.xtc -n {ndx_input} -fit rot+trans -o traj_fit.xtc',
    shell=True, check=True
)

## Index File Generation

In [ ]:
process = subprocess.run(
    ["gmx", "make_ndx", "-f", gro_file, "-n", ndx_input, "-o", ndx_updated],
    input=make_ndx_commands,
    text=True,
    capture_output=True
)

## Plotting Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

### Core Analysis Functions

In [ ]:
import pandas as pd

def load_gro_atom_mapping(gro_filename):
    atom_map = {}
    with open(gro_filename, 'r') as f:
        lines = f.readlines()
    for line in lines[2:-1]:
        if len(line) < 20: continue
        try:
            res_num = line[0:5].strip()
            res_name = line[5:10].strip()
            atom_name = line[10:15].strip()
            atom_num = int(line[15:20].strip())
            atom_map[atom_num] = f"{res_num}{res_name}@{atom_name}"
        except ValueError:
            continue
    return atom_map

def load_hbond_pairs_from_ndx(ndx_filename):
    pairs = {}
    current_idx = 0
    in_target_section = False
    with open(ndx_filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith('['):
                if 'hbonds' in line.lower(): in_target_section = True
                else: in_target_section = False
                continue
            if in_target_section:
                parts = line.split()
                if len(parts) >= 3:
                    pairs[current_idx] = (int(parts[1]), int(parts[2]))
                    current_idx += 1
    return pairs

def parse_xpm(filename):
    with open(filename, 'r') as f:
        lines = f.readlines()
    data_lines = [line.split('"')[1] for line in lines if line.startswith('"') and len(line.split('"')) > 1]
    meta = data_lines[0].split()
    y_dim = int(meta[1])
    num_colors = int(meta[2])
    matrix_lines = data_lines[1 + num_colors:]
    binary_matrix = np.zeros((y_dim, len(matrix_lines[0])), dtype=int)
    for i, line in enumerate(matrix_lines):
        for j, char in enumerate(line):
            if char != ' ':
                binary_matrix[i, j] = 1
    return binary_matrix

### H-Bond Matrix Analysis

In [ ]:
def analyze_hbond_matrix(gro_file=gro_file, ndx_file="hbond.ndx", xpm_file="hbmat.xpm", cutoff=20.0):
    gro_atoms = load_gro_atom_mapping(gro_file)
    atom_mapping = load_hbond_pairs_from_ndx(ndx_file)
    hb_matrix_full = parse_xpm(xpm_file)
    occupancy = np.mean(hb_matrix_full, axis=1) * 100
    valid_indices = [i for i, occ in enumerate(occupancy) if occ > cutoff]
    if len(valid_indices) == 0:
        return pd.DataFrame()
    hb_matrix_filtered = hb_matrix_full[valid_indices, :]
    report_data = []
    yticklabels_labels = []
    for idx, i in enumerate(valid_indices):
        occ = occupancy[i]
        h_atom_idx, a_atom_idx = atom_mapping.get(i, (None, None))
        h_label = gro_atoms.get(h_atom_idx, f"Atom_{h_atom_idx}")
        a_label = gro_atoms.get(a_atom_idx, f"Atom_{a_atom_idx}")
        report_data.append({"bond_idx": i, "occupancy": occ, "hydrogen_label": h_label, "acceptor_label": a_label})
        yticklabels_labels.append(f"Idx {i} ({h_label} ... {a_label})")
    df_report = pd.DataFrame(report_data)
    sort_order = np.argsort(df_report["occupancy"])[::-1]
    df_report = df_report.iloc[sort_order].reset_index(drop=True)
    hb_matrix_filtered = hb_matrix_filtered[sort_order, :]
    yticklabels_labels = [yticklabels_labels[j] for j in sort_order]
    plt.figure(figsize=(14, 2 + len(valid_indices) * 0.4))
    ax = plt.gca()
    sns.heatmap(hb_matrix_filtered, cmap="Blues", cbar=False, yticklabels=yticklabels_labels, ax=ax)
    plt.title(f"H-Bond Existence Map (Occupancy > {cutoff}%)")
    plt.xlabel("Frame")
    plt.ylabel("Hydrogen Bond (H ... A)")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    return df_report

### Protein-Ligand H-Bond Analysis (MDAnalysis)

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis
from IPython.display import display

def analyze_protein_ligand_hbonds(u, protein_resids, ligand_resnames, d_a_cutoff=3.5, d_h_a_angle_cutoff=150):
    protein_query = " or ".join([f"resid {r}" for r in protein_resids])
    ligand_query = " or ".join([f"resname {l}" for l in ligand_resnames])
    hydrogens_prot = f"({protein_query}) and name H*"
    acceptors_lig = f"({ligand_query}) and (name O* or name N*)"
    hydrogens_lig = f"({ligand_query}) and name H*"
    acceptors_prot = f"({protein_query}) and (name O* or name N*)"
    donors_prot = f"({protein_query}) and (name O* or name N*)"
    donors_lig = f"({ligand_query}) and (name O* or name N*)"
    hb_round_A = HydrogenBondAnalysis(universe=u, donors_sel=donors_prot, hydrogens_sel=hydrogens_prot, acceptors_sel=acceptors_lig, d_a_cutoff=d_a_cutoff, d_h_a_angle_cutoff=d_h_a_angle_cutoff, update_selections=False)
    hb_round_A.run()
    hb_round_B = HydrogenBondAnalysis(universe=u, donors_sel=donors_lig, hydrogens_sel=hydrogens_lig, acceptors_sel=acceptors_prot, d_a_cutoff=d_a_cutoff, d_h_a_angle_cutoff=d_h_a_angle_cutoff, update_selections=False)
    hb_round_B.run()
    total_frames = len(u.trajectory)
    bond_counts = {}
    for hb_instance in [hb_round_A, hb_round_B]:
        if hb_instance.results.hbonds is not None and len(hb_instance.results.hbonds) > 0:
            for bond in hb_instance.results.hbonds:
                frame, d_idx, h_idx, a_idx, dist, ang = bond
                pair = (int(d_idx), int(a_idx))
                if pair not in bond_counts: bond_counts[pair] = 0
                bond_counts[pair] += 1
    report = []
    for (d_idx, a_idx), freq in bond_counts.items():
        donor = u.atoms[d_idx]; acceptor = u.atoms[a_idx]
        occ = (freq / total_frames) * 100
        report.append({'Donor': f"{donor.resid}{donor.resname}@{donor.name}", 'Acceptor': f"{acceptor.resid}{acceptor.resname}@{acceptor.name}", 'Frequency': freq, 'Occupancy (%)': occ})
    df_hb = pd.DataFrame(report)
    if not df_hb.empty:
        df_hb = df_hb.sort_values(by='Occupancy (%)', ascending=False).reset_index(drop=True)
        display(df_hb.head(20))
    return df_hb

### Distance Calculation and Plotting Utilities

In [ ]:
import re
from tqdm.auto import tqdm

def parse_gro_mapping(gro_path="step5_1.gro"):
    gro_mapping = {}
    atom_idx_0_based = 0
    with open(gro_path, "r") as f:
        lines = f.readlines()
        for line in lines[2:-1]:
            if len(line.strip()) < 20: continue
            res_num = line[0:5].strip(); res_name = line[5:10].strip(); atom_name = line[10:15].strip()
            gro_mapping[f"{res_num}{res_name}@{atom_name}"] = atom_idx_0_based
            atom_idx_0_based += 1
    return gro_mapping

def calculate_distances_by_indices(u, gro_mapping, bonds_list):
    valid_pairs = []
    for d_label, a_label in bonds_list:
        if d_label not in gro_mapping or a_label not in gro_mapping: continue
        valid_pairs.append((gro_mapping[d_label], gro_mapping[a_label], f"{d_label} ... {a_label}"))
    distances_dict = {label: [] for _, _, label in valid_pairs}
    times = []
    for ts in tqdm(u.trajectory, desc="Траектория"):
        times.append(ts.time)
        for idx1, idx2, label in valid_pairs:
            distances_dict[label].append(np.linalg.norm(u.atoms[idx1].position - u.atoms[idx2].position))
    return np.array(times)/1000.0, distances_dict

def plot_individual_bonds(times, distances_dict, title_prefix="Cluster"):
    labels = list(distances_dict.keys()); n_plots = len(labels)
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3.5 * n_plots), sharex=True)
    if n_plots == 1: axes = [axes]
    for i, label in enumerate(labels):
        ax = axes[i]; raw_dist = np.array(distances_dict[label])
        smoothed_dist = pd.Series(raw_dist).rolling(window=10, min_periods=1).mean()
        ax.plot(times, raw_dist, color='gray', alpha=0.25, linewidth=0.8)
        ax.plot(times, smoothed_dist, color='tab:blue', alpha=0.9, linewidth=1.5)
        ax.axhline(y=3.5, color='tab:red', linestyle='--', linewidth=1.5)
        ax.set_title(label, fontsize=11, loc='left', color='dimgray')
        ax.set_ylabel("Distance (Å)"); ax.set_ylim(2.0, 5.5); ax.grid(True, linestyle=':', alpha=0.5)
    axes[-1].set_xlabel("Time (ns)")
    plt.tight_layout(); plt.show()

def df_to_bond_list(df, min_occupancy=0.0):
    if df is None or df.empty: return []
    filtered_df = df[df['Occupancy (%)'] >= min_occupancy]
    return list(zip(filtered_df['Donor'], filtered_df['Acceptor']))

## GROMACS H-Bond Generation

In [ ]:
subprocess.run(
    f'echo "{hbond_groups}" | gmx hbond -f 4_traj_fit.xtc -s {tpr_file} -n {ndx_updated} -num hbnum.xvg -dist hbdist.xvg -hbm hbmat.xpm -hbn hbond.ndx -nomerge',
    shell=True, check=True
)

## Run H-Bond Matrix Analysis

In [ ]:
results_df = analyze_hbond_matrix()

## Load Universe and RMSD

In [ ]:
u = mda.Universe(tpr_file, xtc_file)

def parse_ndx(filename):
    groups = {}; current_group = None
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('['):
                current_group = line.strip('[] ').strip(); groups[current_group] = []
            elif current_group and line:
                groups[current_group].extend([int(x) - 1 for x in line.split()])
    return groups

ndx_groups = parse_ndx(ndx_updated)
selections = {
    "Protein": "protein",
    "Group 26 (non-Water)": "index " + " ".join(map(str, ndx_groups["non-Water"])),
    "Group 27 (ROH_4GA_0GA_PHO)": "index " + " ".join(map(str, ndx_groups["M6P"])),
    "Group 28 (Complex)": "index " + " ".join(map(str, ndx_groups["Protein_NAD"]))
}
rmsd_results = {}
for name, sel in selections.items():
    R = rms.RMSD(u, u, select=sel, superposition_selection="protein", ref_frame=0)
    R.run()
    rmsd_results[name] = R.results.rmsd

## Bond Distance Analysis

In [ ]:
bond1_sel = "resid 170 and resname ASH and name OD2"
bond2_sel = "resid 263 and resname TYD and name OH"
bond1_partner = "resid 448 and resname 4GA and name O4"
bond2_partner = "resid 449 and resname 0GA and name H2"
b1_dist = []; b2_dist = []; times = []
for ts in tqdm(u.trajectory):
    b1_dist.append(np.linalg.norm(u.select_atoms(bond1_sel)[0].position - u.select_atoms(bond1_partner)[0].position))
    b2_dist.append(np.linalg.norm(u.select_atoms(bond2_sel)[0].position - u.select_atoms(bond2_partner)[0].position))
    times.append(ts.time)
b1_dist = np.array(b1_dist); b2_dist = np.array(b2_dist); times = np.array(times)

## Protein-Ligand H-Bonds (PHO and Sugars)

In [ ]:
my_resids = protein_resids
df_pho = analyze_protein_ligand_hbonds(u, my_resids, ["PHO"])
df_sugars = analyze_protein_ligand_hbonds(u, my_resids, ["4GA", "0GA", "ROH"])
bonds_group_1 = df_to_bond_list(df_pho, min_occupancy=5.0)
bonds_group_2 = df_to_bond_list(df_sugars, min_occupancy=5.0)

## Distance Plots for Clusters

In [ ]:
mapping = parse_gro_mapping("step5_1.gro")
times_g1, dists_g1 = calculate_distances_by_indices(u, mapping, bonds_group_1)
if dists_g1: plot_individual_bonds(times_g1, dists_g1, title_prefix="PHO & NAD Cluster")
times_g2, dists_g2 = calculate_distances_by_indices(u, mapping, bonds_group_2)
if dists_g2: plot_individual_bonds(times_g2, dists_g2, title_prefix="4GA, 0GA & ROH Cluster")

## Ordered Water Analysis

In [ ]:
protein_query = " or ".join([f"resid {r}" for r in protein_resids])
donors_prot = f"({protein_query}) and (name O* or name N*)"
hydrogens_prot = f"({protein_query}) and name H*"
acceptors_prot = f"({protein_query}) and (name O* or name N*)"
donors_wat = f"resname {water_resname} and (name O* or name N*)"
hydrogens_wat = f"resname {water_resname} and name H*"
acceptors_wat = f"resname {water_resname} and (name O* or name N*)"
hb_round_A = HydrogenBondAnalysis(universe=u, donors_sel=donors_prot, hydrogens_sel=hydrogens_prot, acceptors_sel=acceptors_wat, d_a_cutoff=3.5, d_h_a_angle_cutoff=150, update_selections=False)
hb_round_A.run(verbose=True)
hb_round_B = HydrogenBondAnalysis(universe=u, donors_sel=donors_wat, hydrogens_sel=hydrogens_wat, acceptors_sel=acceptors_prot, d_a_cutoff=3.5, d_h_a_angle_cutoff=150, update_selections=False)
hb_round_B.run(verbose=True)
total_frames = len(u.trajectory); bond_counts = {}
for bond in hb_round_A.results.hbonds:
    frame, d_idx, h_idx, a_idx, dist, ang = bond
    w_resid = u.atoms[int(a_idx)].resid
    key = (int(d_idx), w_resid, 'Донор (отдает H воде)')
    bond_counts.setdefault(key, set()).add(int(frame))
for bond in hb_round_B.results.hbonds:
    frame, d_idx, h_idx, a_idx, dist, ang = bond
    w_resid = u.atoms[int(d_idx)].resid
    key = (int(a_idx), w_resid, 'Акцептор (принимает H от воды)')
    bond_counts.setdefault(key, set()).add(int(frame))
report = []
for (p_idx, w_resid, role), frames in bond_counts.items():
    occ = (len(frames) / total_frames) * 100
    if occ >= cutoff_occ:
        atom = u.atoms[p_idx]
        report.append({'Occupancy (%)': occ, 'Protein Atom': f"{atom.resid}{atom.resname}@{atom.name}", 'Water ID': f"{water_resname}_{w_resid}", 'Role': role})
df_hb = pd.DataFrame(report)
if not df_hb.empty: df_hb = df_hb.sort_values(by=['Protein Atom', 'Occupancy (%)'], ascending=[True, False]).reset_index(drop=True)

## Combined Plots

In [ ]:
all_plots = list(rmsd_results.items()) + [("Bond: ASH 170 (OD2) -- 4GA 448 (O4)", b1_dist), ("Bond: TYD 263 (OH) -- 0GA 449 (H2)", b2_dist)]
fig, axes = plt.subplots(len(all_plots), 1, figsize=(12, 3 * len(all_plots)), sharex=False)
for i, (title, data) in enumerate(all_plots):
    ax = axes[i]
    if "Bond" in title and "Hydrogen" not in title:
        ax.plot(times, data, color='firebrick'); ax.set_ylabel("Distance (Å)")
    elif "Hydrogen" in title:
        ax.plot(times, data, color='forestgreen'); ax.set_ylabel("Count")
    else:
        ax.plot(data[:, 1], data[:, 2], color='navy'); ax.set_ylabel("RMSD (Å)")
    ax.set_title(title); ax.grid(True, linestyle='--', alpha=0.6)
axes[-1].set_xlabel("Time (ps)"); plt.tight_layout(); plt.show()

## PDB Trajectory Export

In [ ]:
subprocess.run(f'echo "0" | gmx trjconv -f traj_fit.xtc -s {tpr_file} -n {ndx_updated} -skip 20 -o trajectory.pdb', shell=True, check=True)
subprocess.run(f'echo "38" | gmx trjconv -f traj_fit.xtc -s {tpr_file} -n {ndx_updated} -o trajectory_no_h20.pdb', shell=True, check=True)

## PyMOL Visualization Setup

In [ ]:
import __main__
__main__.pymol_argv = ['pymol', '-x']
import pymol
pymol.finish_launching()
from pymol import cmd
cmd.do('load trajectory.pdb, my_traj')
cmd.do('remove solvent and not resi 3846')
cmd.do('delete all')
cmd.do('load trajectory_no_h20.pdb, my_traj')

### PyMOL Interaction Builders

In [ ]:
import re
def parse_atom(atom_string):
    res_part, atom_name = atom_string.split('@')
    match = re.match(r"^(\d+)_?([A-Za-z0-9]+)$", res_part)
    resi, resn = match.groups()
    atom_sel = f"resi {resi} and resn {resn} and name \"{atom_name}\""
    res_sel = f"resi {resi} and resn {resn}"
    return atom_sel, res_sel

def build_interactions(group_name, bonds):
    residues_to_select = set(); dist_objects = []
    for i, (a1_str, a2_str) in enumerate(bonds):
        atom1_sel, res1_sel = parse_atom(a1_str)
        atom2_sel, res2_sel = parse_atom(a2_str)
        residues_to_select.add(res1_sel); residues_to_select.add(res2_sel)
        dist_name = f"{group_name}_dist_{i+1}"
        cmd.distance(dist_name, atom1_sel, atom2_sel); dist_objects.append(dist_name)
    if residues_to_select:
        cmd.select(f"{group_name}_residues", " or ".join(residues_to_select))
    if dist_objects: cmd.group(f"{group_name}_measurements", " ".join(dist_objects))

build_interactions("Group1", bonds_group_1)
build_interactions("Group2", bonds_group_2)

def build_mn_cluster():
    cmd.select("Mn_complex_residues", "resi 169 or resi 200 or resn 0GA or resn MN or resn NAD")
    mn_sel = "resn MN"
    coordination_targets = {"Mn_S_link": "(resi 169 or resi 200) and name SG", "Mn_NE2_link": "(resi 169 or resi 200) and name NE2", "Mn_O2_sugar": "(resn 0GA or resn NAD) and name O2", "Mn_O3_sugar": "(resn 0GA or resn NAD) and name O3", "Mn_O7N_link": "(resn 0GA or resn NAD) and name O7N"}
    dist_objects = []
    for dist_name, target_sel in coordination_targets.items():
        cmd.distance(dist_name, mn_sel, target_sel); dist_objects.append(dist_name)
    if dist_objects: cmd.group("Mn_coordination_bonds", " ".join(dist_objects))
build_mn_cluster()

def build_catalytic_contacts():
    cmd.select("Catalytic_residues", "resi 170 or resi 263 or resi 448 or resi 449")
    cmd.distance("Tyr263_0GA_link", "resi 263 and name OH", "resi 449 and resn 0GA and name H2")
    cmd.distance("Ash170_4GA_link", "resi 170 and resn ASH and name HD2", "resi 448 and resn 4GA and name O4")
    cmd.group("Ligand_H_bonds", "Tyr263_0GA_link Ash170_4GA_link")
build_catalytic_contacts()

def build_water_glu_contact_fixed():
    wat_sel = "(resi 3846 and resn WAT)"; view_sel = "(resi 109 or resi 110)"
    cmd.select("Wat_Glu_residues_Fixed", f"{wat_sel} or {view_sel}")
    cmd.distance("Wat3846_Glu109_link_Fixed", f"{wat_sel} and (name O or name OW)", "(resi 109 and resn GLU) and (name OE1 or name OE2)")
    cmd.group("Solvent_contacts_Fixed", "Wat3846_Glu109_link_Fixed")
build_water_glu_contact_fixed()